# Week 6: Probability & Distributions

# Applied Statistics with Python (2026)

---

Last week we described data we already have. This week we study the **mathematical models** that generate data.  
Probability theory is the language of uncertainty — and distributions are its vocabulary.

> "Probability is the very guide of life." — Cicero

### What We'll Cover

| # | Topic |
|---|-------|
| 1 | Basic Probability Rules |
| 2 | Conditional Probability |
| 3 | Bayes' Theorem |
| 4 | Random Variables & Expectation |
| 5 | Discrete Distributions — Bernoulli & Binomial |
| 6 | Discrete Distributions — Poisson |
| 7 | Continuous Distributions — Uniform |
| 8 | Continuous Distributions — Normal (Gaussian) |
| 9 | Continuous Distributions — Exponential |
| 10 | Working with `scipy.stats` |
| 11 | Checking Normality |
| 12 | Practical Example: Simulating Real-World Scenarios |

Import all necessary libraries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Plot settings
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (8, 5)
rng = np.random.default_rng(2026)  # reproducible randomness

print("Libraries loaded!")

---
## 1. Basic Probability Rules

Probability measures **how likely** an event is to occur, on a scale from 0 (impossible) to 1 (certain).

$$P(A) = \frac{\text{Number of favorable outcomes}}{\text{Total number of outcomes}}$$

### Key Rules

| Rule | Formula | Meaning |
|------|---------|--------|
| Complement | $P(A^c) = 1 - P(A)$ | Probability of NOT A |
| Addition (OR) | $P(A \cup B) = P(A) + P(B) - P(A \cap B)$ | A or B (or both) |
| Multiplication (AND) | $P(A \cap B) = P(A) \cdot P(B \mid A)$ | A and B together |
| Independent events | $P(A \cap B) = P(A) \cdot P(B)$ | If A and B don't affect each other |

### 1.1 Simulating Coin Flips

Let's verify probability rules through simulation — the **frequentist** approach: run many trials and observe the proportion.

In [ ]:
# Simulate 10,000 fair coin flips
n_flips = 10_000
flips = rng.choice(["H", "T"], size=n_flips)  # H = Heads, T = Tails

# Count outcomes
n_heads = (flips == "H").sum()
n_tails = (flips == "T").sum()

print(f"Flipped {n_flips} coins:")
print(f"  Heads: {n_heads} ({n_heads/n_flips:.4f})")
print(f"  Tails: {n_tails} ({n_tails/n_flips:.4f})")
print(f"  Theoretical: 0.5000")

### 1.2 Simulating Dice Rolls

Verify the addition rule: P(rolling a 1 OR a 6) = P(1) + P(6) = 1/6 + 1/6 = 2/6 ≈ 0.333.

In [ ]:
# Simulate 10,000 dice rolls
n_rolls = 10_000
rolls = rng.integers(1, 7, size=n_rolls)  # integers from 1 to 6

# P(1 or 6)
p_1_or_6 = ((rolls == 1) | (rolls == 6)).mean()
print(f"P(1 or 6) = {p_1_or_6:.4f}  (theoretical: {2/6:.4f})")

# P(even) = P(2 or 4 or 6)
p_even = (rolls % 2 == 0).mean()
print(f"P(even)   = {p_even:.4f}  (theoretical: 0.5000)")

# P(>= 5) = P(5 or 6)
p_ge5 = (rolls >= 5).mean()
print(f"P(≥ 5)    = {p_ge5:.4f}  (theoretical: {2/6:.4f})")

# Distribution of outcomes
print(f"\nFrequency distribution:")
for face in range(1, 7):
    count = (rolls == face).sum()
    print(f"  {face}: {count:>5} ({count/n_rolls:.3f})")

### 1.3 The Law of Large Numbers

As the number of trials increases, the observed proportion **converges** to the true probability. Let's watch it happen.

In [ ]:
# Law of Large Numbers: coin flip convergence
n = 5000
flips = rng.choice([0, 1], size=n)  # 0=Tails, 1=Heads

# Running proportion of heads after each flip
running_prop = np.cumsum(flips) / np.arange(1, n + 1)

# Plot convergence
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(running_prop, color="steelblue", linewidth=0.8)
ax.axhline(0.5, color="red", linestyle="--", linewidth=2, label="True P(H) = 0.5")
ax.set_xlabel("Number of Flips")
ax.set_ylabel("Proportion of Heads")
ax.set_title("Law of Large Numbers: Coin Flip Convergence")
ax.set_ylim(0.3, 0.7)
ax.legend()
plt.tight_layout()
plt.show()

---
## 2. Conditional Probability

Conditional probability is the probability of event A **given that** event B has already occurred:

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}$$

### Example
In a class of 80 students, if 30 are CS majors and 12 of those CS majors got an A, then:
$$P(A \mid CS) = \frac{12}{30} = 0.40$$

Let's compute conditional probabilities from a student dataset.

In [ ]:
# Create a student dataset
n = 200
students = pd.DataFrame({
    "Major": rng.choice(["CS", "Statistics", "Math", "Economics"], size=n,
                         p=[0.35, 0.25, 0.20, 0.20]),
    "Grade": rng.choice(["A", "B", "C", "D", "F"], size=n,
                         p=[0.15, 0.30, 0.30, 0.15, 0.10]),
    "Gender": rng.choice(["Male", "Female"], size=n),
})

# Cross-tabulation
cross = pd.crosstab(students["Major"], students["Grade"], margins=True)
print("=== Major × Grade ===")
print(cross)

Compute various conditional probabilities from the cross-tabulation.

In [ ]:
# P(A | CS) — probability of grade A given CS major
cs_students = students[students["Major"] == "CS"]
p_a_given_cs = (cs_students["Grade"] == "A").mean()
print(f"P(Grade=A | Major=CS) = {p_a_given_cs:.4f}")

# P(CS | A) — probability of being CS given grade A
a_students = students[students["Grade"] == "A"]
p_cs_given_a = (a_students["Major"] == "CS").mean()
print(f"P(Major=CS | Grade=A) = {p_cs_given_a:.4f}")

# Note: P(A|CS) ≠ P(CS|A) — this is a common mistake!
print(f"\n⚠️  P(A|CS) ≠ P(CS|A): {p_a_given_cs:.4f} ≠ {p_cs_given_a:.4f}")

# P(A or B | CS)
p_ab_given_cs = cs_students["Grade"].isin(["A", "B"]).mean()
print(f"P(Grade=A or B | Major=CS) = {p_ab_given_cs:.4f}")

# Conditional probabilities for all majors
print("\n=== P(Grade=A | Major=X) for each Major ===")
for major in students["Major"].unique():
    subset = students[students["Major"] == major]
    p_a = (subset["Grade"] == "A").mean()
    print(f"  P(A | {major:12s}) = {p_a:.4f}  (n={len(subset)})")

### 2.1 Independence Test

Two events A and B are **independent** if knowing B doesn't change the probability of A:  
$P(A \mid B) = P(A)$  

Let's check: is getting an A independent of Major?

In [ ]:
# Is Grade independent of Major?
p_a_overall = (students["Grade"] == "A").mean()
print(f"P(Grade=A) overall = {p_a_overall:.4f}")

print("\nIf independent, P(A|Major) should ≈ P(A) for all majors:")
for major in sorted(students["Major"].unique()):
    subset = students[students["Major"] == major]
    p_a = (subset["Grade"] == "A").mean()
    diff = abs(p_a - p_a_overall)
    status = "≈ independent" if diff < 0.05 else "possibly dependent"
    print(f"  P(A|{major:12s}) = {p_a:.4f}  (diff={diff:.4f}) → {status}")

---
## 3. Bayes' Theorem

Bayes' theorem lets us **reverse** conditional probabilities — going from P(B|A) to P(A|B):

$$P(A \mid B) = \frac{P(B \mid A) \cdot P(A)}{P(B)}$$

Where:
- $P(A)$ = **prior** — what we believed before seeing evidence
- $P(B \mid A)$ = **likelihood** — probability of evidence given A is true
- $P(A \mid B)$ = **posterior** — updated belief after seeing evidence
- $P(B)$ = **evidence** — total probability of B

### Classic Example: Medical Testing

A disease affects 1% of the population. A test has:
- **Sensitivity** (true positive rate): 95% — correctly detects sick people
- **Specificity** (true negative rate): 90% — correctly identifies healthy people

If you test positive, what's the probability you actually have the disease?

Solve the medical testing problem step by step using Bayes' theorem.

In [ ]:
# Bayes' Theorem: Medical Test Example
p_disease = 0.01            # P(D) — prior: 1% have the disease
p_healthy = 1 - p_disease   # P(H) = 0.99
sensitivity = 0.95          # P(+|D) — true positive rate
specificity = 0.90          # P(-|H) — true negative rate
false_positive = 1 - specificity  # P(+|H) = 0.10

# P(+) = P(+|D)·P(D) + P(+|H)·P(H)   (total probability)
p_positive = sensitivity * p_disease + false_positive * p_healthy

# P(D|+) = P(+|D)·P(D) / P(+)   (Bayes' theorem)
p_disease_given_positive = (sensitivity * p_disease) / p_positive

print("=== Medical Test: Bayes' Theorem ===")
print(f"P(Disease)           = {p_disease:.4f}")
print(f"P(+|Disease)         = {sensitivity:.4f}")
print(f"P(+|Healthy)         = {false_positive:.4f}")
print(f"P(+)                 = {p_positive:.4f}")
print(f"")
print(f"P(Disease | Positive) = {p_disease_given_positive:.4f}")
print(f"")
print(f"→ Only {p_disease_given_positive*100:.1f}% of positive tests actually have the disease!")
print(f"→ This is the 'base rate fallacy' — when the disease is rare,")
print(f"  even a good test produces many false positives.")

Verify Bayes' theorem result through simulation — simulate a population and count.

In [ ]:
# Simulation verification of Bayes' theorem
population = 100_000

# Simulate who has the disease (1% prevalence)
has_disease = rng.random(population) < p_disease

# Simulate test results
test_positive = np.zeros(population, dtype=bool)
test_positive[has_disease] = rng.random(has_disease.sum()) < sensitivity     # true positives
test_positive[~has_disease] = rng.random((~has_disease).sum()) < false_positive  # false positives

# Count outcomes
true_pos = (test_positive & has_disease).sum()
false_pos = (test_positive & ~has_disease).sum()
total_pos = test_positive.sum()

print(f"=== Simulation ({population:,} people) ===")
print(f"Actually sick:     {has_disease.sum():,}")
print(f"Tested positive:   {total_pos:,}")
print(f"  True positives:  {true_pos:,}")
print(f"  False positives: {false_pos:,}")
print(f"")
print(f"P(Disease|+) = {true_pos}/{total_pos} = {true_pos/total_pos:.4f}")
print(f"Bayes' formula:                          {p_disease_given_positive:.4f}")

Visualize how the posterior changes with disease prevalence.

In [ ]:
# How does P(Disease|+) change with prevalence?
prevalences = np.linspace(0.001, 0.20, 200)
posteriors = []
for prev in prevalences:
    p_pos = sensitivity * prev + false_positive * (1 - prev)
    posterior = (sensitivity * prev) / p_pos
    posteriors.append(posterior)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(prevalences * 100, np.array(posteriors) * 100, linewidth=2, color="steelblue")
ax.axhline(50, color="gray", linestyle="--", alpha=0.5, label="50% threshold")
ax.axvline(1, color="red", linestyle="--", alpha=0.5, label="1% prevalence")
ax.scatter([1], [p_disease_given_positive * 100], color="red", s=100, zorder=5)
ax.set_xlabel("Disease Prevalence (%)")
ax.set_ylabel("P(Disease | Positive Test) (%)")
ax.set_title("Bayes' Theorem: Posterior vs Prevalence")
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Random Variables & Expectation

A **random variable** X assigns a numerical value to each outcome of a random experiment.

| Concept | Discrete | Continuous |
|---------|----------|------------|
| Probability | $P(X = x)$ | $f(x)$ (density function) |
| Expected value (mean) | $E[X] = \sum x \cdot P(x)$ | $E[X] = \int x \cdot f(x)\,dx$ |
| Variance | $Var(X) = E[(X - \mu)^2]$ | Same formula |
| Std deviation | $\sigma = \sqrt{Var(X)}$ | Same formula |

Compute the expected value and variance of a die roll manually and via simulation.

In [ ]:
# Expected value of a fair die
outcomes = np.array([1, 2, 3, 4, 5, 6])
probabilities = np.array([1/6] * 6)  # each face equally likely

# E[X] = sum of x * P(x)
expected_value = np.sum(outcomes * probabilities)
print(f"E[X] = {expected_value:.4f}  (theoretical: 3.5)")

# Var(X) = E[(X - mu)^2]
variance = np.sum((outcomes - expected_value)**2 * probabilities)
std_dev = np.sqrt(variance)
print(f"Var(X) = {variance:.4f}  (theoretical: {35/12:.4f})")
print(f"Std(X) = {std_dev:.4f}")

# Verify with simulation
simulated_rolls = rng.integers(1, 7, size=100_000)
print(f"\nSimulation (100,000 rolls):")
print(f"  Mean = {simulated_rolls.mean():.4f}")
print(f"  Std  = {simulated_rolls.std():.4f}")

---
## 5. Discrete Distributions — Bernoulli & Binomial

### Bernoulli Distribution
A single trial with two outcomes: **success** (1) or **failure** (0).
$$P(X = 1) = p, \quad P(X = 0) = 1-p$$
$$E[X] = p, \quad Var(X) = p(1-p)$$

### Binomial Distribution
The number of successes in **n independent** Bernoulli trials.
$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}$$
$$E[X] = np, \quad Var(X) = np(1-p)$$

**Use when:** counting successes out of a fixed number of independent trials (coin flips, defective items, pass/fail).

### 5.1 Bernoulli — Single Trial

Simulate Bernoulli trials: each student has a 70% chance of passing an exam.

In [ ]:
# Bernoulli distribution: pass/fail
p_pass = 0.70  # 70% pass rate

# Using scipy.stats
bernoulli_rv = stats.bernoulli(p=p_pass)
print(f"Bernoulli(p={p_pass}):")
print(f"  E[X] = {bernoulli_rv.mean():.2f}")
print(f"  Var(X) = {bernoulli_rv.var():.4f}")
print(f"  P(X=0) = {bernoulli_rv.pmf(0):.2f}")
print(f"  P(X=1) = {bernoulli_rv.pmf(1):.2f}")

# Simulate 20 students
results = bernoulli_rv.rvs(size=20, random_state=2026)
print(f"\n20 students: {results}")
print(f"Passed: {results.sum()}/{len(results)} = {results.mean():.2f}")

### 5.2 Binomial — Multiple Trials

If 70% of students pass, and we have a class of 30, how many will pass? The binomial distribution answers this.

In [ ]:
# Binomial distribution: n=30 trials, p=0.70
n_students = 30
p_pass = 0.70

binom_rv = stats.binom(n=n_students, p=p_pass)

print(f"Binomial(n={n_students}, p={p_pass}):")
print(f"  E[X] = {binom_rv.mean():.1f}")
print(f"  Std(X) = {binom_rv.std():.2f}")
print(f"")

# Probability questions
print(f"  P(X = 21) = {binom_rv.pmf(21):.4f}  (exactly 21 pass)")
print(f"  P(X ≥ 25) = {1 - binom_rv.cdf(24):.4f}  (25 or more pass)")
print(f"  P(X ≤ 15) = {binom_rv.cdf(15):.4f}  (15 or fewer pass)")
print(f"  P(X < 20) = {binom_rv.cdf(19):.4f}  (fewer than 20 pass)")

Visualize the binomial PMF (probability mass function) — shows the probability of each possible outcome.

In [ ]:
# Binomial PMF visualization
k_values = np.arange(0, n_students + 1)
pmf_values = binom_rv.pmf(k_values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PMF
colors = ["coral" if k < 20 else "steelblue" for k in k_values]
axes[0].bar(k_values, pmf_values, color=colors, edgecolor="white")
axes[0].axvline(binom_rv.mean(), color="red", linestyle="--", label=f"Mean={binom_rv.mean():.1f}")
axes[0].set_xlabel("Number of Students Passing (k)")
axes[0].set_ylabel("P(X = k)")
axes[0].set_title(f"Binomial PMF — B({n_students}, {p_pass})")
axes[0].legend()

# CDF
cdf_values = binom_rv.cdf(k_values)
axes[1].step(k_values, cdf_values, color="steelblue", linewidth=2)
axes[1].axhline(0.5, color="gray", linestyle="--", alpha=0.5)
axes[1].set_xlabel("k")
axes[1].set_ylabel("P(X ≤ k)")
axes[1].set_title(f"Binomial CDF — B({n_students}, {p_pass})")

plt.tight_layout()
plt.show()

Compare binomial distributions with different parameters.

In [ ]:
# Compare different binomial distributions
fig, ax = plt.subplots(figsize=(10, 5))

params = [(30, 0.3), (30, 0.5), (30, 0.7), (30, 0.9)]
for n, p in params:
    rv = stats.binom(n=n, p=p)
    k = np.arange(0, n + 1)
    ax.plot(k, rv.pmf(k), "o-", label=f"B({n}, {p})", markersize=3)

ax.set_xlabel("k (number of successes)")
ax.set_ylabel("P(X = k)")
ax.set_title("Binomial Distributions with Different p")
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Discrete Distributions — Poisson

The Poisson distribution models the number of **rare events** in a fixed interval of time or space.

$$P(X = k) = \frac{\lambda^k e^{-\lambda}}{k!}$$

$$E[X] = \lambda, \quad Var(X) = \lambda$$

**Key property:** Mean equals variance ($\lambda$).

**Use when:** counting events per unit (emails per hour, accidents per month, typos per page).

A university help desk receives an average of 5 questions per hour. What's the probability of receiving exactly 3? More than 8?

In [ ]:
# Poisson distribution: help desk questions
lam = 5  # average 5 questions per hour

poisson_rv = stats.poisson(mu=lam)

print(f"Poisson(λ={lam}):")
print(f"  E[X] = {poisson_rv.mean():.1f}")
print(f"  Var(X) = {poisson_rv.var():.1f}  (note: mean = variance)")
print(f"  Std(X) = {poisson_rv.std():.2f}")
print(f"")
print(f"  P(X = 3) = {poisson_rv.pmf(3):.4f}  (exactly 3 questions)")
print(f"  P(X = 5) = {poisson_rv.pmf(5):.4f}  (exactly 5 questions)")
print(f"  P(X ≤ 3) = {poisson_rv.cdf(3):.4f}  (3 or fewer)")
print(f"  P(X > 8) = {1 - poisson_rv.cdf(8):.4f}  (more than 8)")
print(f"  P(X = 0) = {poisson_rv.pmf(0):.4f}  (no questions at all)")

Visualize Poisson distributions with different rates ($\lambda$).

In [ ]:
# Compare Poisson distributions with different λ
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Different λ values
lambdas = [1, 3, 5, 10, 15]
k_max = 25
k = np.arange(0, k_max + 1)

for lam in lambdas:
    rv = stats.poisson(mu=lam)
    axes[0].plot(k, rv.pmf(k), "o-", label=f"λ={lam}", markersize=3)

axes[0].set_xlabel("k")
axes[0].set_ylabel("P(X = k)")
axes[0].set_title("Poisson PMF for Different λ")
axes[0].legend()

# Simulation vs theory for λ=5
simulated = rng.poisson(lam=5, size=10_000)
axes[1].hist(simulated, bins=range(0, 20), density=True,
             alpha=0.6, color="steelblue", edgecolor="white", label="Simulated")
axes[1].plot(k[:18], stats.poisson.pmf(k[:18], 5), "ro-",
             markersize=6, label="Theoretical")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Probability")
axes[1].set_title("Poisson(λ=5): Simulation vs Theory")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 7. Continuous Distributions — Uniform

The **Uniform distribution** gives **equal probability** to all values in an interval $[a, b]$.

$$f(x) = \frac{1}{b-a} \quad \text{for } a \leq x \leq b$$

$$E[X] = \frac{a+b}{2}, \quad Var(X) = \frac{(b-a)^2}{12}$$

**Use when:** all outcomes are equally likely (random number generators, random selection).

Simulate and visualize a Uniform(0, 10) distribution.

In [ ]:
# Uniform distribution
a, b = 0, 10
uniform_rv = stats.uniform(loc=a, scale=b - a)  # scipy uses loc, scale

print(f"Uniform({a}, {b}):")
print(f"  E[X] = {uniform_rv.mean():.1f}")
print(f"  Std(X) = {uniform_rv.std():.2f}")
print(f"  P(X ≤ 3) = {uniform_rv.cdf(3):.2f}")
print(f"  P(2 ≤ X ≤ 7) = {uniform_rv.cdf(7) - uniform_rv.cdf(2):.2f}")

# Visualization
fig, ax = plt.subplots(figsize=(8, 4))
x = np.linspace(-1, 11, 300)

# Theoretical PDF
ax.plot(x, uniform_rv.pdf(x), "r-", linewidth=2, label="PDF (theoretical)")

# Simulated data
samples = uniform_rv.rvs(size=5000, random_state=2026)
ax.hist(samples, bins=30, density=True, alpha=0.5, color="steelblue",
        edgecolor="white", label="Simulated (n=5000)")

ax.set_title(f"Uniform({a}, {b})")
ax.set_xlabel("x")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.show()

---
## 8. Continuous Distributions — Normal (Gaussian)

The **Normal distribution** is the most important distribution in statistics.  
It's the famous bell curve, completely defined by its mean ($\mu$) and standard deviation ($\sigma$).

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

### The 68-95-99.7 Rule (Empirical Rule)

| Range | Probability |
|-------|-------------|
| $\mu \pm 1\sigma$ | ≈ 68% |
| $\mu \pm 2\sigma$ | ≈ 95% |
| $\mu \pm 3\sigma$ | ≈ 99.7% |

### Why Is It So Important?
1. Many natural phenomena are approximately normal (heights, IQ, measurement errors)
2. The **Central Limit Theorem** says sample means approach normal regardless of the original distribution
3. Many statistical tests assume normality

### 8.1 Standard Normal (Z) Distribution

The standard normal has $\mu = 0$ and $\sigma = 1$. Any normal can be converted to standard normal via: $Z = \frac{X - \mu}{\sigma}$.

In [ ]:
# Standard Normal distribution
z_rv = stats.norm(loc=0, scale=1)

print("Standard Normal N(0, 1):")
print(f"  P(Z ≤ 0)     = {z_rv.cdf(0):.4f}")
print(f"  P(Z ≤ 1.96)  = {z_rv.cdf(1.96):.4f}")
print(f"  P(Z > 1.96)  = {1 - z_rv.cdf(1.96):.4f}")
print(f"  P(-1.96 ≤ Z ≤ 1.96) = {z_rv.cdf(1.96) - z_rv.cdf(-1.96):.4f}  ← 95%!")
print(f"")
print(f"Empirical Rule:")
print(f"  P(-1σ ≤ Z ≤ +1σ) = {z_rv.cdf(1) - z_rv.cdf(-1):.4f}  ← ≈68%")
print(f"  P(-2σ ≤ Z ≤ +2σ) = {z_rv.cdf(2) - z_rv.cdf(-2):.4f}  ← ≈95%")
print(f"  P(-3σ ≤ Z ≤ +3σ) = {z_rv.cdf(3) - z_rv.cdf(-3):.4f}  ← ≈99.7%")

Visualize the 68-95-99.7 rule with shaded regions.

In [ ]:
# 68-95-99.7 Rule visualization
fig, ax = plt.subplots(figsize=(10, 5))
x = np.linspace(-4, 4, 500)
y = z_rv.pdf(x)

# Plot the bell curve
ax.plot(x, y, "k-", linewidth=2)

# Shade regions
ax.fill_between(x, y, where=(x >= -3) & (x <= 3), alpha=0.15, color="green", label="99.7% (±3σ)")
ax.fill_between(x, y, where=(x >= -2) & (x <= 2), alpha=0.20, color="blue", label="95% (±2σ)")
ax.fill_between(x, y, where=(x >= -1) & (x <= 1), alpha=0.30, color="red", label="68% (±1σ)")

# Labels
for z in [-3, -2, -1, 0, 1, 2, 3]:
    ax.axvline(z, color="gray", linewidth=0.5, linestyle=":")
    ax.text(z, -0.02, f"{z}σ", ha="center", fontsize=9)

ax.set_title("The 68-95-99.7 Rule (Empirical Rule)", fontsize=14)
ax.set_xlabel("Standard Deviations from Mean")
ax.set_ylabel("Density")
ax.legend(loc="upper right")
ax.set_ylim(-0.05, 0.45)
plt.tight_layout()
plt.show()

### 8.2 Working with Normal Distributions

Exam scores follow N(72, 12). Compute probabilities and find score cutoffs.

In [ ]:
# Exam scores: N(mu=72, sigma=12)
mu, sigma = 72, 12
exam_rv = stats.norm(loc=mu, scale=sigma)

print(f"Exam Scores ~ N({mu}, {sigma}):")
print(f"")

# Forward: X → Probability
print("--- Probabilities (CDF) ---")
print(f"  P(Score ≤ 60)  = {exam_rv.cdf(60):.4f}")
print(f"  P(Score ≥ 90)  = {1 - exam_rv.cdf(90):.4f}")
print(f"  P(60 ≤ X ≤ 84) = {exam_rv.cdf(84) - exam_rv.cdf(60):.4f}")

# Reverse: Probability → X (quantile / percent point function)
print(f"")
print("--- Cutoff Scores (PPF / Inverse CDF) ---")
print(f"  Top 10% cutoff:    {exam_rv.ppf(0.90):.1f}")
print(f"  Top 5% cutoff:     {exam_rv.ppf(0.95):.1f}")
print(f"  Bottom 10% cutoff: {exam_rv.ppf(0.10):.1f}")
print(f"  Median:            {exam_rv.ppf(0.50):.1f}")

Compare normal distributions with different means and standard deviations.

In [ ]:
# Compare normal distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Different means, same std
x = np.linspace(20, 120, 500)
for mu in [50, 65, 80]:
    axes[0].plot(x, stats.norm.pdf(x, mu, 12), label=f"μ={mu}, σ=12", linewidth=2)
axes[0].set_title("Different Means, Same Spread")
axes[0].set_xlabel("x")
axes[0].set_ylabel("Density")
axes[0].legend()

# Same mean, different std
for sigma in [5, 10, 20]:
    axes[1].plot(x, stats.norm.pdf(x, 70, sigma), label=f"μ=70, σ={sigma}", linewidth=2)
axes[1].set_title("Same Mean, Different Spread")
axes[1].set_xlabel("x")
axes[1].set_ylabel("Density")
axes[1].legend()

plt.tight_layout()
plt.show()

### 8.3 Z-Score Conversion

Convert any normal variable to the standard normal using: $Z = \frac{X - \mu}{\sigma}$. This lets you compare values from different scales.

In [ ]:
# Z-score conversion example
# Alice scored 88 on Midterm (mean=72, std=12)
# Alice scored 85 on Final (mean=70, std=16)
# Which performance was relatively better?

midterm_score, midterm_mu, midterm_sigma = 88, 72, 12
final_score, final_mu, final_sigma = 85, 70, 16

z_midterm = (midterm_score - midterm_mu) / midterm_sigma
z_final = (final_score - final_mu) / final_sigma

print("Alice's Performance:")
print(f"  Midterm: {midterm_score} → z = {z_midterm:.2f} → top {(1 - stats.norm.cdf(z_midterm))*100:.1f}%")
print(f"  Final:   {final_score} → z = {z_final:.2f} → top {(1 - stats.norm.cdf(z_final))*100:.1f}%")
print(f"")
if z_midterm > z_final:
    print(f"→ Midterm was relatively better (higher z-score)")
else:
    print(f"→ Final was relatively better (higher z-score)")

---
## 9. Continuous Distributions — Exponential

The **Exponential distribution** models the **time between** events in a Poisson process.

$$f(x) = \lambda e^{-\lambda x}, \quad x \geq 0$$

$$E[X] = \frac{1}{\lambda}, \quad Var(X) = \frac{1}{\lambda^2}$$

If events arrive at rate $\lambda$ per unit time (Poisson), then the waiting time between events is Exponential($\lambda$).

**Use when:** modeling wait times, time between events, survival analysis.

If the help desk receives 5 questions per hour (Poisson λ=5), what's the distribution of time between questions?

In [ ]:
# Exponential distribution: time between help desk questions
lam = 5  # 5 questions per hour
# scipy parameterizes with scale = 1/λ
exp_rv = stats.expon(scale=1/lam)  # scale = 1/5 = 0.2 hours = 12 minutes

print(f"Exponential(λ={lam}):")
print(f"  Mean wait time: {exp_rv.mean()*60:.1f} minutes")
print(f"  Std:            {exp_rv.std()*60:.1f} minutes")
print(f"")
print(f"  P(wait ≤ 6 min)  = {exp_rv.cdf(6/60):.4f}")
print(f"  P(wait ≤ 12 min) = {exp_rv.cdf(12/60):.4f}")
print(f"  P(wait > 20 min) = {1 - exp_rv.cdf(20/60):.4f}")

Visualize exponential distributions with different rates.

In [ ]:
# Compare exponential distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.linspace(0, 3, 300)

# Different λ values
for lam in [0.5, 1, 2, 5]:
    rv = stats.expon(scale=1/lam)
    axes[0].plot(x, rv.pdf(x), label=f"λ={lam}", linewidth=2)

axes[0].set_xlabel("x (time)")
axes[0].set_ylabel("Density")
axes[0].set_title("Exponential PDF for Different λ")
axes[0].legend()

# Simulation vs theory for λ=5
simulated = rng.exponential(scale=1/5, size=5000)
axes[1].hist(simulated * 60, bins=40, density=True, alpha=0.6,
             color="steelblue", edgecolor="white", label="Simulated")
x_min = np.linspace(0, 60, 300)
axes[1].plot(x_min, stats.expon.pdf(x_min / 60, scale=1/5) / 60,
             "r-", linewidth=2, label="Theoretical")
axes[1].set_xlabel("Wait Time (minutes)")
axes[1].set_ylabel("Density")
axes[1].set_title("Time Between Questions (λ=5/hr)")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 10. Working with `scipy.stats`

All `scipy.stats` distributions share a **consistent API** — learn it once, use it everywhere.

| Method | What It Does | Example |
|--------|--------------|---------|
| `rv.pmf(k)` | P(X=k) — **discrete** only | `binom.pmf(3, n=10, p=0.5)` |
| `rv.pdf(x)` | Density at x — **continuous** only | `norm.pdf(0, loc=0, scale=1)` |
| `rv.cdf(x)` | P(X ≤ x) — cumulative distribution | `norm.cdf(1.96)` |
| `rv.sf(x)` | P(X > x) = 1 − CDF — survival function | `norm.sf(1.96)` |
| `rv.ppf(q)` | Inverse CDF — find x for given probability | `norm.ppf(0.975)` |
| `rv.rvs(size)` | Generate random samples | `norm.rvs(size=1000)` |
| `rv.mean()` | Expected value | `binom(n=10, p=0.5).mean()` |
| `rv.var()` | Variance | `poisson(mu=5).var()` |
| `rv.interval(α)` | Central interval containing α probability | `norm.interval(0.95)` |

A unified demonstration showing the same API across different distributions.

In [ ]:
# Unified API demonstration
distributions = {
    "Binomial(20, 0.6)": stats.binom(n=20, p=0.6),
    "Poisson(λ=7)":      stats.poisson(mu=7),
    "Normal(70, 10)":    stats.norm(loc=70, scale=10),
    "Exponential(λ=2)": stats.expon(scale=1/2),
    "Uniform(0, 100)":   stats.uniform(loc=0, scale=100),
}

print(f"{'Distribution':<22} {'Mean':>8} {'Std':>8} {'Median':>8} {'P(X≤mean)':>10}")
print("-" * 60)
for name, rv in distributions.items():
    print(f"{name:<22} {rv.mean():>8.2f} {rv.std():>8.2f} {rv.median():>8.2f} {rv.cdf(rv.mean()):>10.4f}")

Generate random samples and compare with theoretical distributions.

In [ ]:
# Generate samples and compare with theory
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Binomial
binom_rv = stats.binom(n=20, p=0.6)
samples = binom_rv.rvs(size=5000, random_state=42)
k = np.arange(0, 21)
axes[0, 0].hist(samples, bins=np.arange(-0.5, 21.5, 1), density=True,
                alpha=0.6, color="steelblue", edgecolor="white", label="Simulated")
axes[0, 0].plot(k, binom_rv.pmf(k), "ro-", markersize=5, label="Theoretical")
axes[0, 0].set_title(f"Binomial(20, 0.6)  μ={binom_rv.mean():.1f}")
axes[0, 0].legend()

# Poisson
pois_rv = stats.poisson(mu=7)
samples = pois_rv.rvs(size=5000, random_state=42)
k = np.arange(0, 20)
axes[0, 1].hist(samples, bins=np.arange(-0.5, 20.5, 1), density=True,
                alpha=0.6, color="mediumseagreen", edgecolor="white", label="Simulated")
axes[0, 1].plot(k, pois_rv.pmf(k), "ro-", markersize=5, label="Theoretical")
axes[0, 1].set_title(f"Poisson(λ=7)  μ={pois_rv.mean():.1f}")
axes[0, 1].legend()

# Normal
norm_rv = stats.norm(loc=70, scale=10)
samples = norm_rv.rvs(size=5000, random_state=42)
x = np.linspace(30, 110, 300)
axes[1, 0].hist(samples, bins=40, density=True, alpha=0.6,
                color="coral", edgecolor="white", label="Simulated")
axes[1, 0].plot(x, norm_rv.pdf(x), "r-", linewidth=2, label="Theoretical")
axes[1, 0].set_title(f"Normal(70, 10)  μ={norm_rv.mean():.1f}")
axes[1, 0].legend()

# Exponential
exp_rv = stats.expon(scale=1/2)
samples = exp_rv.rvs(size=5000, random_state=42)
x = np.linspace(0, 4, 300)
axes[1, 1].hist(samples, bins=40, density=True, alpha=0.6,
                color="orchid", edgecolor="white", label="Simulated")
axes[1, 1].plot(x, exp_rv.pdf(x), "r-", linewidth=2, label="Theoretical")
axes[1, 1].set_title(f"Exponential(λ=2)  μ={exp_rv.mean():.2f}")
axes[1, 1].legend()

plt.suptitle("Simulation vs Theoretical Distributions", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 11. Checking Normality

Many statistical tests require data to be **approximately normal**. How do we check?

| Method | Type | Interpretation |
|--------|------|----------------|
| **Histogram + KDE** | Visual | Bell-shaped? Symmetric? |
| **Q-Q Plot** | Visual | Points follow the diagonal line? |
| **Shapiro-Wilk Test** | Statistical | p > 0.05 → can't reject normality |
| **Skewness / Kurtosis** | Numerical | Close to 0 / 0? |

### 11.1 Q-Q Plot

A Q-Q (quantile-quantile) plot compares data quantiles against theoretical normal quantiles. If data is normal, points fall along the diagonal.

In [ ]:
# Generate different data types for comparison
normal_data = rng.normal(70, 10, 200)
skewed_data = rng.exponential(10, 200)
uniform_data = rng.uniform(40, 100, 200)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

datasets = [
    (normal_data, "Normal Data"),
    (skewed_data, "Skewed Data (Exponential)"),
    (uniform_data, "Uniform Data"),
]

for ax, (data, title) in zip(axes, datasets):
    stats.probplot(data, dist="norm", plot=ax)
    ax.set_title(f"Q-Q Plot: {title}")
    ax.get_lines()[0].set_markerfacecolor("steelblue")
    ax.get_lines()[0].set_alpha(0.5)

plt.tight_layout()
plt.show()

### 11.2 Shapiro-Wilk Test

A formal statistical test for normality. The null hypothesis is: "the data is normally distributed."

In [ ]:
# Shapiro-Wilk normality test
datasets = {
    "Normal data":     normal_data,
    "Exponential data": skewed_data,
    "Uniform data":    uniform_data,
}

print("=== Shapiro-Wilk Normality Test ===")
print("H0: Data is normally distributed")
print("If p < 0.05 → reject H0 (not normal)")
print("")

for name, data in datasets.items():
    stat, p_value = stats.shapiro(data)
    skew = pd.Series(data).skew()
    kurt = pd.Series(data).kurtosis()
    verdict = "✓ Normal" if p_value > 0.05 else "✗ Not Normal"
    print(f"  {name:22s}  W={stat:.4f}  p={p_value:.4f}  skew={skew:+.2f}  → {verdict}")

### 11.3 Combined Normality Check

Let's build a reusable function that performs a complete normality check: histogram + Q-Q plot + Shapiro-Wilk test.

In [ ]:
def check_normality(data, name="Variable"):
    """Perform a complete normality check: histogram, Q-Q plot, Shapiro-Wilk."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Histogram + KDE + theoretical normal
    sns.histplot(data, kde=True, ax=axes[0], color="steelblue", stat="density")
    x = np.linspace(data.min(), data.max(), 200)
    axes[0].plot(x, stats.norm.pdf(x, data.mean(), data.std()),
                "r--", linewidth=2, label="Normal fit")
    axes[0].set_title(f"{name}: Histogram + KDE")
    axes[0].legend()
    
    # Q-Q plot
    stats.probplot(data, dist="norm", plot=axes[1])
    axes[1].set_title(f"{name}: Q-Q Plot")
    
    plt.tight_layout()
    plt.show()
    
    # Shapiro-Wilk test
    stat, p = stats.shapiro(data)
    skew = pd.Series(data).skew()
    kurt = pd.Series(data).kurtosis()
    print(f"  Shapiro-Wilk: W={stat:.4f}, p={p:.4f}")
    print(f"  Skewness: {skew:.3f},  Kurtosis: {kurt:.3f}")
    print(f"  Verdict: {'Normal ✓' if p > 0.05 else 'Not Normal ✗'}")

# Test with exam score data
df = pd.read_csv("students_eda.csv")
check_normality(df["Midterm"].dropna().values, "Midterm Scores")

Check normality for another variable that might be skewed.

In [ ]:
# Check StudyHours — likely not normal (right-skewed)
check_normality(df["StudyHours"].dropna().values, "Study Hours")

---
## 12. Practical Example: Simulating Real-World Scenarios

Let's use distributions to **model and simulate** realistic scenarios — the bridge between theory and practice.

### Scenario 1: Exam Pass Rates (Binomial)

In a class of 40, the historical pass rate is 75%. If we run 1000 simulated classes, what does the distribution of pass counts look like?

In [ ]:
# Scenario 1: Simulating pass counts across many classes
n_class = 40
p_pass = 0.75
n_simulations = 1000

# Simulate 1000 classes
pass_counts = rng.binomial(n=n_class, p=p_pass, size=n_simulations)

print(f"Simulated {n_simulations} classes of {n_class} students (p={p_pass}):")
print(f"  Average passes:  {pass_counts.mean():.1f}  (expected: {n_class * p_pass})")
print(f"  Std dev:         {pass_counts.std():.2f}  (expected: {np.sqrt(n_class*p_pass*(1-p_pass)):.2f})")
print(f"  Range:           [{pass_counts.min()}, {pass_counts.max()}]")
print(f"  P(all 40 pass):  {(pass_counts == 40).mean():.4f}")
print(f"  P(≤ 25 pass):    {(pass_counts <= 25).mean():.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(pass_counts, bins=range(15, 42), density=True,
        alpha=0.6, color="steelblue", edgecolor="white", label="Simulated")
k = np.arange(15, 41)
ax.plot(k, stats.binom.pmf(k, n_class, p_pass), "ro-",
        markersize=4, label="Theoretical Binomial")
ax.axvline(n_class * p_pass, color="red", linestyle="--", label=f"E[X]={n_class*p_pass}")
ax.set_xlabel("Number of Students Passing")
ax.set_ylabel("Probability")
ax.set_title("Scenario 1: How Many Students Pass? (B(40, 0.75))")
ax.legend()
plt.tight_layout()
plt.show()

### Scenario 2: Server Requests (Poisson)

A web server gets an average of 20 requests per minute. Simulate one hour of traffic and analyze the pattern.

In [ ]:
# Scenario 2: Server request simulation
lam_per_minute = 20
n_minutes = 60

# Simulate requests per minute for 1 hour
requests = rng.poisson(lam=lam_per_minute, size=n_minutes)

print(f"Server Traffic Simulation ({n_minutes} minutes, λ={lam_per_minute}/min):")
print(f"  Total requests:     {requests.sum()}")
print(f"  Average per minute: {requests.mean():.1f}")
print(f"  Max in one minute:  {requests.max()}")
print(f"  Min in one minute:  {requests.min()}")
print(f"  Minutes with > 30:  {(requests > 30).sum()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Time series of requests
axes[0].bar(range(n_minutes), requests, color="steelblue", alpha=0.7)
axes[0].axhline(lam_per_minute, color="red", linestyle="--", label=f"λ={lam_per_minute}")
axes[0].set_xlabel("Minute")
axes[0].set_ylabel("Requests")
axes[0].set_title("Requests Per Minute (1 Hour)")
axes[0].legend()

# Distribution
axes[1].hist(requests, bins=15, density=True, alpha=0.6,
             color="steelblue", edgecolor="white", label="Observed")
k = np.arange(5, 40)
axes[1].plot(k, stats.poisson.pmf(k, lam_per_minute), "ro-",
             markersize=4, label="Poisson(20)")
axes[1].set_xlabel("Requests per Minute")
axes[1].set_ylabel("Probability")
axes[1].set_title("Request Count Distribution")
axes[1].legend()

plt.tight_layout()
plt.show()

### Scenario 3: Grading on a Curve (Normal)

A professor wants to assign grades so that scores follow a fixed distribution: top 10% get A, next 20% get B, etc. Use the normal distribution to find the cutoffs.

In [ ]:
# Scenario 3: Grading on a curve
mu, sigma = 68, 15  # class average 68, std 15
exam = stats.norm(loc=mu, scale=sigma)

# Grade distribution: A(10%), B(20%), C(40%), D(20%), F(10%)
cutoffs = {
    "A (top 10%)": exam.ppf(0.90),
    "B (top 30%)": exam.ppf(0.70),
    "C (top 70%)": exam.ppf(0.30),
    "D (top 90%)": exam.ppf(0.10),
}

print(f"Class: N(μ={mu}, σ={sigma})")
print(f"\n{'Grade':<15} {'Cutoff Score':>14}")
print("-" * 30)
print(f"{'A (top 10%)':<15} {'≥ ' + f'{cutoffs["A (top 10%)"]:.1f}':>14}")
print(f"{'B (next 20%)':<15} {'≥ ' + f'{cutoffs["B (top 30%)"]:.1f}':>14}")
print(f"{'C (next 40%)':<15} {'≥ ' + f'{cutoffs["C (top 70%)"]:.1f}':>14}")
print(f"{'D (next 20%)':<15} {'≥ ' + f'{cutoffs["D (top 90%)"]:.1f}':>14}")
print(f"{'F (bottom 10%)':<15} {'< ' + f'{cutoffs["D (top 90%)"]:.1f}':>14}")

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 500)
y = exam.pdf(x)

# Color regions
grade_colors = {"F": "#d32f2f", "D": "#ff9800", "C": "#fdd835", "B": "#66bb6a", "A": "#1976d2"}
bounds = [mu - 4*sigma, cutoffs["D (top 90%)"], cutoffs["C (top 70%)"],
          cutoffs["B (top 30%)"], cutoffs["A (top 10%)"], mu + 4*sigma]
labels = ["F", "D", "C", "B", "A"]

for i, label in enumerate(labels):
    mask = (x >= bounds[i]) & (x <= bounds[i+1])
    ax.fill_between(x[mask], y[mask], alpha=0.6, color=grade_colors[label], label=label)

ax.plot(x, y, "k-", linewidth=2)
ax.set_xlabel("Score")
ax.set_ylabel("Density")
ax.set_title("Grading on a Curve")
ax.legend(title="Grade")
plt.tight_layout()
plt.show()

### Scenario 4: Distribution Comparison Summary

A final visual comparing all the distributions we learned side by side.

In [ ]:
# Distribution comparison chart
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Bernoulli
p = 0.7
axes[0, 0].bar([0, 1], [1-p, p], color=["coral", "steelblue"], edgecolor="white")
axes[0, 0].set_xticks([0, 1])
axes[0, 0].set_xticklabels(["Fail (0)", "Pass (1)"])
axes[0, 0].set_title(f"Bernoulli(p={p})")
axes[0, 0].set_ylabel("Probability")

# 2. Binomial
k = np.arange(0, 21)
axes[0, 1].bar(k, stats.binom.pmf(k, 20, 0.6), color="steelblue", edgecolor="white")
axes[0, 1].set_title("Binomial(n=20, p=0.6)")
axes[0, 1].set_xlabel("k")

# 3. Poisson
k = np.arange(0, 20)
axes[0, 2].bar(k, stats.poisson.pmf(k, 5), color="mediumseagreen", edgecolor="white")
axes[0, 2].set_title("Poisson(λ=5)")
axes[0, 2].set_xlabel("k")

# 4. Uniform
x = np.linspace(-1, 11, 300)
axes[1, 0].plot(x, stats.uniform.pdf(x, 0, 10), linewidth=2, color="orchid")
axes[1, 0].fill_between(x, stats.uniform.pdf(x, 0, 10), alpha=0.3, color="orchid")
axes[1, 0].set_title("Uniform(0, 10)")
axes[1, 0].set_xlabel("x")

# 5. Normal
x = np.linspace(30, 110, 300)
axes[1, 1].plot(x, stats.norm.pdf(x, 70, 10), linewidth=2, color="coral")
axes[1, 1].fill_between(x, stats.norm.pdf(x, 70, 10), alpha=0.3, color="coral")
axes[1, 1].set_title("Normal(μ=70, σ=10)")
axes[1, 1].set_xlabel("x")

# 6. Exponential
x = np.linspace(0, 5, 300)
axes[1, 2].plot(x, stats.expon.pdf(x, scale=0.5), linewidth=2, color="teal")
axes[1, 2].fill_between(x, stats.expon.pdf(x, scale=0.5), alpha=0.3, color="teal")
axes[1, 2].set_title("Exponential(λ=2)")
axes[1, 2].set_xlabel("x")

for ax in axes.flatten():
    ax.set_ylabel("Probability / Density")

plt.suptitle("Distribution Zoo: All Distributions at a Glance", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 13. Summary

| # | Topic | Key Concepts |
|---|-------|-------------|
| 1 | Probability Rules | Complement, addition, multiplication, independence |
| 2 | Conditional Probability | $P(A \mid B)$, $P(A \mid B) \neq P(B \mid A)$ |
| 3 | Bayes' Theorem | Prior → Likelihood → Posterior, base rate fallacy |
| 4 | Random Variables | Expected value $E[X]$, variance $Var(X)$ |
| 5 | Binomial | $B(n,p)$ — # successes in n trials, `stats.binom` |
| 6 | Poisson | $Pois(\lambda)$ — events per interval, mean=variance, `stats.poisson` |
| 7 | Uniform | $U(a,b)$ — equal probability, `stats.uniform` |
| 8 | Normal | $N(\mu,\sigma)$ — bell curve, 68-95-99.7 rule, z-scores, `stats.norm` |
| 9 | Exponential | $Exp(\lambda)$ — wait times, memoryless, `stats.expon` |
| 10 | `scipy.stats` API | `.pmf/.pdf`, `.cdf`, `.ppf`, `.rvs`, `.mean()`, `.var()` |
| 11 | Normality Check | Histogram, Q-Q plot, Shapiro-Wilk test |

### Distribution Decision Guide

| Question | Distribution |
|----------|-------------|
| How many successes in n trials? | **Binomial** |
| How many events per time period? | **Poisson** |
| What's a random value in a range? | **Uniform** |
| Where does a measurement fall? | **Normal** |
| How long until the next event? | **Exponential** |

---
## 14. Homework (Week 6)

### Task 1: Probability & Bayes
1. A factory produces 5% defective items. A quality test catches 98% of defects but also flags 3% of good items. If an item is flagged, what's the probability it's actually defective? (Use Bayes' theorem)
2. Verify your answer with a simulation of 100,000 items
3. Plot how the posterior changes as the defect rate varies from 0.1% to 20%

### Task 2: Binomial & Poisson
1. A multiple-choice quiz has 20 questions, each with 4 choices. If a student guesses randomly:
   - What's the expected number of correct answers?
   - What's P(getting ≥ 10 correct)?
   - Plot the PMF
2. A bus arrives every 10 minutes on average. Model arrivals per hour with Poisson and:
   - Find P(more than 8 buses in an hour)
   - Find P(no bus for 20 minutes) using Exponential

### Task 3: Normal Distribution
1. IQ scores follow N(100, 15). Find:
   - P(IQ > 130) — "gifted" threshold
   - The IQ score at the 99th percentile
   - P(85 < IQ < 115) — "average" range
2. Plot the PDF with shaded regions for each answer above
3. Generate 1000 random IQ scores and verify your answers

### Task 4: Normality Testing
1. Load `students_eda.csv` from Week 5
2. For each numeric column: create a Q-Q plot and run the Shapiro-Wilk test
3. Which variables are approximately normal? Which are not?
4. For non-normal variables, suggest a possible distribution that fits better

---
## Next Week Preview

**Week 7: Sampling & Estimation** — From samples to populations:
- Sampling methods (random, stratified, systematic)
- The Central Limit Theorem (CLT) — the most important theorem in statistics
- Confidence intervals
- Bootstrap estimation

Now that you understand probability distributions, it's time to use them for **inference** — drawing conclusions about a population from a sample!

---
*Applied Statistics with Python (2026) | Week 6 | Probability & Distributions*